### LÄSA IN DATA

In [21]:
import sqlite3

In [22]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# players_df = pd.read_csv('players.csv')
# appearances_df = pd.read_csv('appearances.csv')

# df = pd.merge(appearances_df, players_df, on='player_id', how='left')

# # print(df.head())
# # df.info()


# y = df['market_value_in_eur']
# X = df.drop(columns=['market_value_in_eur'])

# print(X.columns)

# features = ['goals', 'assists', 'minutes_played', 'position','']

import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# # 1. Läs in och summera
# players = pd.read_csv('players.csv')
# apps = pd.read_csv('appearances.csv')

# stats = (
#     apps.groupby('player_id')[['goals', 'assists', 'minutes_played']]
#     .sum() 
#     .reset_index()
# )

# # 2. Slå ihop och välj kolumner
# df = pd.merge(players, stats, on='player_id').dropna(
#     subset=['market_value_in_eur']
# )
# df['age'] = 2026 - pd.to_datetime(df['date_of_birth']).dt.year

# features = [
#     'position', 
#     'age', 
#     'goals', 
#     'assists', 
#     'minutes_played',
#     'current_club_domestic_competition_id'
#     ]

# target = 'market_value_in_eur'

# conn = sqlite3.connect('football.db')
# df_from_db = pd.read_sql("SELECT * FROM players_data", conn)
# conn.close()

# df_final = df[features + [target]].copy()

# # 3. X och y (get_dummies gör om position direkt)
#X = pd.get_dummies(df_from_db[features], drop_first=True) 

#y = df_from_db[target]

#print(df_from_db.shape)


In [33]:
import os
import pandas as pd
from dotenv import load_dotenv
from pymongo import MongoClient

# 1. Läs in miljövariabler från .env
load_dotenv()
mongo_uri = os.getenv("MONGODB_URI")

if not mongo_uri:
    raise ValueError("Kunde inte hitta MONGO_URI. Kontrollera att .env-filen finns och innehåller rätt variabel.")

# 2. Koppla upp mot MongoDB Atlas
client = MongoClient(mongo_uri)
db = client["football_data"]
collection = db["players_data"]

# 3. Hämta alla dokument och uteslut _id
data = list(collection.find({}, {"_id": 0}))

# 4. Gör om till DataFrame
df_from_db = pd.DataFrame(data)

# 5. Bygg X och y
target = 'market_value_in_eur'
y = df_from_db[target]
X_features = df_from_db.drop(columns=[target])

X = pd.get_dummies(X_features, drop_first=True)

print(df_from_db.shape)

(28328, 7)


### EDA

In [43]:

print(y.mean())

1991035.3713640214


In [23]:
# import sqlite3
# import pandas as pd
# 
# conn = sqlite3.connect("football.db")
# 
# tables = pd.read_sql_query(
#     "SELECT name FROM sqlite_master WHERE type='table';",
#     conn
# )
# 
# tables
# conn = sqlite3.connect('football1.db')
# df_from_db = pd.read_sql("SELECT * FROM players_data", conn)
# conn.close()

# df_final = df[features + [target]].copy()

# # # # 3. X och y (get_dummies gör om position direkt)
# X = pd.get_dummies(df_from_db, drop_first=True) 

# #y = df_from_db[target]

# print(df_from_db.shape)

# läsa in från mongo db

In [32]:
# 1. Kolla om vi faktiskt fick några rader
print(f"Antal rader hämtade: {len(data)}")

# 2. Kolla vilka kolumner som faktiskt finns i df
print("Befintliga kolumner:")
print(df_from_db.columns.tolist())

# 3. Kika på första dokumentet om det finns något
if len(data) > 0:
    print("\nFörsta raden i databasen:")
    print(data[0])

Antal rader hämtade: 0
Befintliga kolumner:
[]


In [ ]:
import pandas as pd
from pymongo import MongoClient

# 1. Koppla upp mot MongoDB (ersätt med din anslutningssträng om ni kör Atlas)
client = MongoClient("mongodb://localhost:27017/")
db = client["football_db"]
collection = db["players_data"]

# 2. Hämta alla dokument och uteslut Mongo-ID:t (_id)
data = list(collection.find({}, {"_id": 0}))

# 3. Gör om listan med dokument till en Pandas DataFrame
df_from_db = pd.DataFrame(data)

# 4. Bygg X och y precis som tidigare
# Kom ihåg att separera target innan du kör get_dummies
target = 'market_value_in_eur'
y = df_from_db[target]
X_features = df_from_db.drop(columns=[target])

X = pd.get_dummies(X_features, drop_first=True)

print(df_from_db.shape)

ServerSelectionTimeoutError: localhost:27017: [WinError 10061] Det gick inte att göra en anslutning eftersom måldatorn aktivt nekade det (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 6aabac8d09cf282ab579c66c, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [WinError 10061] Det gick inte att göra en anslutning eftersom måldatorn aktivt nekade det (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>

In [24]:

# y = df_from_db['market_value_in_eur']
# X = X.drop(columns=['market_value_in_eur'])



In [25]:
# X.current_club_domestic_competition_id_IT1
# #y.head()

### TRÄNA / TESTA

In [34]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score
X_train_full, X_test, y_train_full, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42)

X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

age_imputer = SimpleImputer(strategy='mean')       # eller 'median'
age_imputer.fit(X_train[['age']])                   # lär sig BARA från X_train

X_train['age'] = age_imputer.transform(X_train[['age']]) 
X_val['age']   = age_imputer.transform(X_val[['age']])
X_test['age']  = age_imputer.transform(X_test[['age']])

#Ny 
y_train_log = np.log1p(y_train)

params = {
    'max_depth': [None, 5, 10, 20, 30],
    'n_estimators': [50, 100, 150],
    'min_samples_split': [2, 5, 10]
}
clf = RandomForestRegressor(random_state=42)
gs = GridSearchCV(estimator=clf, param_grid=params, cv=2, n_jobs=-1, verbose=2)
gs.fit(X_train, y_train_log)
print("Bästa parametrar:", gs.best_params_) 
models = {
    "BaseLine" : DummyRegressor(strategy='mean'),
    'Linjär Regression' : LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    "Random Forest (Tuned)": gs.best_estimator_
}


for name, model in models.items():
    if name != "Random Forest (Tuned)":
        model.fit(X_train, y_train_log)

    # Prediktera och konvertera tillbaka till vanliga euro
    log_preds = model.predict(X_val)
    preds = np.expm1(log_preds)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)

    print(f'{name:20} | {rmse:12,.0f} € | {r2:6.3f}') ## <---------------------------------------------



# Träna på log(y) istället för y direkt
# y_train_log = np.log1p(y_train)
# model.fit(X_train, y_train_log)

# Transformera tillbaka gissningarna med expm1 innan du räknar RMSE

# preds = np.expm1(model.predict(X_test)).copu


Fitting 2 folds for each of 45 candidates, totalling 90 fits
Bästa parametrar: {'max_depth': 20, 'min_samples_split': 10, 'n_estimators': 150}
BaseLine             |    7,652,756 € | -0.049
Linjär Regression    |    7,230,735 € |  0.064
Random Forest        |    5,122,266 € |  0.530
Random Forest (Tuned) |    5,079,378 € |  0.538


In [ ]:
test_pred_log = gs.best_estimator_.predict(X_test)
test_pred = np.expm1(test_pred_log)

final_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
final_r2 = r2_score(y_test, test_pred)

print('RMSE = ', final_rmse)
print('R2 = ', final_r2)


### SPARA

In [ ]:
# import joblib
# modell = joblib.dump(models['Random Forest'], "rfr.pkl")

# rfr = joblib.load("rfr.pkl")

In [35]:
import joblib
modell = joblib.dump(models['Random Forest (Tuned)'], "rfr_tuned.pkl")

rfr = joblib.load("rfr_tuned.pkl")

print(type(rfr))

<class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [ ]:
X.info()

In [ ]:
import numpy as np
import pandas as pd

# 1. Skapa en tom rad med exakt samma kolumnnamn som träningsdatan
calle = pd.DataFrame(0, index=[0], columns=X_train.columns)

# 2. Fyll i siffervärden
calle['age'] = 18
calle['goals'] = 1000
calle['assists'] = 9200
calle['minutes_played'] = 9200 * 90  # t.ex. en hel säsong som ordinarie

# 3. Sätt en 1:a på hans position (Anfallare / Attack)
# Beroende på hur texten ser ut i er data heter den oftast 'position_Attack'
for col in calle.columns:
    if 'position' in col and 'Attack' in col:
        calle[col] = 1

# 4. Sätt en 1:a på ligan (Allsvenskan = SE1)
for col in calle.columns:
    if 'SE1' in col:
        calle[col] = 1

# 5. Låt modellen gissa (kom ihåg att omvandla från log-skala med expm1!)
basta_modell = gs.best_estimator_
calle_log_pred = basta_modell.predict(calle)
calle_varde = np.expm1(calle_log_pred)[0]

print(f"Spelare: Calle Pålsson")
print(f"Stats:   22 mål, 22 assist, Allsvenskan")
print(f"Uppskattat marknadsvärde: {calle_varde:,.0f} €")